In [ ]:
import MDAnalysis as mda
from mg_sampling.density.forward_density import get_model_map, get_model_map_olly, voxels_around

import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np

import torch

from mg_sampling import PROJECT_DIR

In [ ]:
u = mda.Universe(PROJECT_DIR / 'density/plain_md_7ez0_mMg_tip3p/aligned_residue/aligned_0.pdb', to_guess=('types',))

In [ ]:
u.atoms.types

In [ ]:
ag = u.select_atoms('nucleic and not type H')
ag = u.atoms
len(ag)

In [ ]:
map_params = {'x0': [0, 0, 0], 'nbin': [256, 256, 256], 'dx': [0.8, 0.8, 0.8]}


In [ ]:
voxel_indices = voxels_around(ag, 4, map_params)

In [ ]:
flatmap = get_model_map(ag, chunk_size=20000, device='cuda', vox_indices=voxel_indices, **map_params)

In [ ]:
flatmap = None

In [ ]:
map = np.zeros(map_params['nbin'])

In [ ]:
map = map.flatten(order = 'F')
map[voxel_indices] = flatmap.cpu()
map = map.reshape(map_params['nbin'], order = 'F')

In [ ]:
nbin = map_params['nbin']
dx = map_params['dx']
extent = [0, nbin[0]*dx[0], 0, nbin[0]*dx[0]]
plt.imshow(map[126, :, :].T, origin='lower', cmap='gnuplot2_r', extent=extent)
#plt.scatter(ag.positions[:, 1], ag.positions[:, 2], s=0.1)

In [ ]:
%timeit map_olly = get_model_map_olly(ag, device='cuda', **map_params)

In [ ]:
extent = [0, nbin[0]*dx[0], 0, nbin[0]*dx[0]]
plt.imshow(map_olly[126, :, :].T, origin='lower', cmap='gnuplot2_r', extent=extent)
#plt.scatter(ag.positions[:, 1], ag.positions[:, 2], s=1)

In [ ]:
plt.imshow(map_olly[156].T - map[156].T, origin='lower')
plt.colorbar()

### Torch 

In [ ]:
from mg_sampling.density.forward_density import get_fmod_param
device = 'cuda'
cutoff = 4
def xyz_to_ijk(xyz, map_params):
    dx = np.asarray(map_params['dx'])  # (3,)
    x0 = np.asarray(map_params['x0'])  # (3,)
    ijk = np.floor((xyz - x0) / dx).astype(int)  # (N, 3)
    return ijk

def ijk_to_xyz(ijk, map_params):
    dx = np.asarray(map_params['dx'])
    x0 = np.asarray(map_params['x0'])
    xyz = x0 + ijk * dx
    return xyz

dx = torch.tensor(map_params['dx']).to(torch.float).to(device)  # (3,)
x0 = torch.tensor(map_params['x0']).to(torch.float).to(device)  # (3,)
map_torch = torch.zeros(*map_params['nbin']).to(device)  # (X, Y, Z)

a_pref, a_invs2 = get_fmod_param(ag)
a_pref, a_invs2 = torch.tensor(a_pref).to(torch.float).to(device), torch.tensor(a_invs2).to(torch.float).to(device)

a_xyz = torch.tensor(ag.positions).to(torch.float).to(device)  # (N, 3)
a_occ = torch.tensor(ag.occupancies).to(torch.float).to(device)  # (N,)

a_ijk = torch.floor((a_xyz - x0) / dx).to(torch.int)  # (N, 3)

oi, oj, ok = torch.ceil(cutoff / dx).to(torch.int)
offsets = torch.arange(-oi, oi+1), torch.arange(-oj, oj+1), torch.arange(-ok, ok+1)
offset_kernel = torch.cartesian_prod(*offsets)  # 3 * (Δ, 3)
av_ijk = a_ijk[:, None] + offset_kernel[None, :]  # (N, Δ, 3)
av_xyz = x0 + av_ijk * dx
av_d2 = torch.sum((av_xyz - a_xyz[:, None])**2, dim=-1)  # (N, Δ)
av_dens = (av_d2 <= cutoff**2) * torch.sum(
    a_pref[:, None, :] * torch.exp(-0.5 * a_invs2[:, None, :] * av_d2[:, :, None]),  # (N, Δ, 5)
    dim=-1) * a_occ[:, None]  # (N, Δ)

v_i, v_j, v_k = av_ijk.movedim(2, 0)  # 3 * (N, Δ)
map_torch.index_put_((v_i, v_j, v_k), av_dens, accumulate=True)

In [ ]:
extent = torch.tensor([0, nbin[0]*dx[0], 0, nbin[0]*dx[0]]).numpy()
plt.imshow(map_torch.cpu().numpy()[126, :, :].T, origin='lower', cmap='gnuplot2_r', extent=extent)
#plt.scatter(ag.positions[:, 1], ag.positions[:, 2], s=1)

In [ ]:
plt.imshow(map_torch.cpu().numpy()[156].T - map[156].T, origin='lower')
plt.colorbar()